#Dataset Dados Abertos STJ - Íntegras de Decisões Terminativas e Acórdãos do Diário da Justiça (Decisões)

##Experimento 13: Remoção de STOP Words e manutenção da pontuação.

##Importação de bibliotecas e declaração de variáveis globais

In [1]:
ENVIRONMENT = 'COLAB' #'PUCPR'

import torch

# Check if a GPU (CUDA) is available
if torch.cuda.is_available():
    print("GPU is available!")
    print("Device Name:", torch.cuda.get_device_name(0))
else:
    print("GPU not available, using CPU.")

# This is the line you'd use in your script to send your model or data to the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

GPU is available!
Device Name: NVIDIA L4
Using device: cuda


##Parametrização conforme ambiente de execução.

In [3]:
import pandas as pd

# Define the base path to the dataset folder
if ENVIRONMENT == 'COLAB':
  from google.colab import drive
  drive.mount('/content/drive')

  base_path = '/content/drive/MyDrive/_Mestrado PUCPR/Datasets/'
  num_layers_to_freeze = 165 # You can adjust this number based on your inspection of layer names
  #model_checkpoint = 'neuralmind/bert-base-portuguese-cased'
  model_checkpoint = 'neuralmind/bert-large-portuguese-cased'
elif ENVIRONMENT == 'PUCPR':
  num_layers_to_freeze = 165 # You can adjust this number based on your inspection of layer names
  base_path = 'dataset/'
  model_checkpoint = 'neuralmind/bert-large-portuguese-cased'
else:
  raise Exception("Environment of execution is not recognize!")

base_path = '/content/drive/MyDrive/_Mestrado PUCPR/Datasets/'
install_evaluate = False

EPOCHS = 150
APPROACH_EXECUTION = 'epoch' # 'step'

##Carregar classe/rótulo para a decisão (Favorável ou Não ao Réu)

In [4]:
# Define file paths
file_path = base_path + 'descricao-decisao-julgamento.csv'

# Load each CSV file into a dataframe
try:
    dfTipoDecisao = pd.read_csv(file_path, sep=';')
    print(f"Successfully loaded {file_path} into dataframe 'dfTipoDecisao'")
    display(dfTipoDecisao.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except Exception as e:
    print(f"An error occurred while loading {file_path}: {e}")

Successfully loaded /content/drive/MyDrive/_Mestrado PUCPR/Datasets/descricao-decisao-julgamento.csv into dataframe 'dfTipoDecisao'


,COD_TEOR_DECISAO,DESC_TEOR_DECISAO,DESC_RSULT_JGTO,RESULTADO_RESUMIDO,DECISAO_FAVORAVEL
0,1,Concedendo,"AGRAVO CONHECIDO EM PARTE E, NESSA PARTE, PROVIDO",PARCIALMENTE CONHECIDO E PROVIDO,S
1,1,Concedendo,Agravo regimental provido,PROVIDO,S
2,1,Concedendo,CONCEDENDO HABEAS CORPUS,CONCEDIDO,S
3,1,Concedendo,CONCEDENDO HABEAS DATA,CONCEDIDO,S
4,1,Concedendo,CONCEDENDO MANDADO DE SEGURANÇA,CONCEDIDA,S


##Carregar metadados das decisões (Documento, Processo, Nro de Registro, Ministro, Teor).

In [5]:
file_path = base_path + 'metadados-decisoes-acordaos-dj052025.csv'

try:
    dfMetaDadosDecisao = pd.read_csv(file_path, sep=';')
    print(f"Successfully loaded {file_path} into dataframe 'dfMetaDadosDecisao'")
    display(dfMetaDadosDecisao.head(3))
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except Exception as e:
    print(f"An error occurred while loading {file_path}: {e}")

Successfully loaded /content/drive/MyDrive/_Mestrado PUCPR/Datasets/metadados-decisoes-acordaos-dj052025.csv into dataframe 'dfMetaDadosDecisao'


,SEQDOCUMENTO,DATAPUBLICACAO,TIPODOCUMENTO,NUMEROREGISTRO,PROCESSO,DATARECEBIMENTO,DATADISTRIBUICAO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,ASSUNTOS
0,309631639,2025-05-05,DECISÃO,202000946481,TP 2695,2020-04-25,2020-04-27,MARIA ISABEL GALLOTTI,NaN,Negando,Prejudicada a ação de #{nome_da_parte} #{campo...,4993
1,310378377,2025-05-05,DECISÃO,202400362531,AREsp 2562174,2024-02-09,2024-02-28,JOÃO OTÁVIO DE NORONHA,NaN,Negando,Conhecido o recurso de #{nome_da_parte} e não-...,"11806, 7752, 7780, 7779"
2,309409286,2025-05-05,DECISÃO,202401080738,REsp 2132967,2024-04-02,2024-04-04,MESSOD AZULAY NETO,NaN,Concedendo,Conhecido o recurso de #{nome_da_parte} e prov...,"3416, 3435"


##Carregar detalhes das decisões

In [6]:
# Define file paths
file_path = base_path + 'decisoes-acordaos-dj052025.csv'

try:
    dfDetalheDecisao = pd.read_csv(file_path, sep=';')
    print(f"Successfully loaded {file_path} into dataframe 'dfDetalheDecisoes'")
    display(dfDetalheDecisao.head(3))
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except Exception as e:
    print(f"An error occurred while loading {file_path}: {e}")

Successfully loaded /content/drive/MyDrive/_Mestrado PUCPR/Datasets/decisoes-acordaos-dj052025.csv into dataframe 'dfDetalheDecisoes'


,SEQDOCUMENTO,ACORDAO,EMENTA,RELATORIO,VOTO,DECISAO
0,309631639,NaN,NaN,NaN,NaN,O presente pedido de contracautela tinha por o...
1,310378377,NaN,NaN,NaN,NaN,Trata-se de agravo em recurso especial interpo...
2,309409286,NaN,NaN,NaN,NaN,Trata-se de recurso especial interposto por MA...


In [7]:
dfMetaDadosDecisao.shape

(48588, 12)

In [8]:
dfDetalheDecisao.shape

(48027, 6)

##Tratamento dos dados
###Exclusão de campos desnecessários

In [9]:
dfMetaDadosDecisao.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48588 entries, 0 to 48587
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   SEQDOCUMENTO          48588 non-null  int64 
 1   DATAPUBLICACAO        48588 non-null  object
 2   TIPODOCUMENTO         48588 non-null  object
 3   NUMEROREGISTRO        48588 non-null  int64 
 4   PROCESSO              48588 non-null  object
 5   DATARECEBIMENTO       48588 non-null  object
 6   DATADISTRIBUICAO      48588 non-null  object
 7   NM_MINISTRO           48588 non-null  object
 8   RECURSO               6200 non-null   object
 9   TEOR                  48588 non-null  object
 10  DESCRICAOMONOCRATICA  48588 non-null  object
 11  ASSUNTOS              48588 non-null  object
dtypes: int64(2), object(10)
memory usage: 4.4+ MB


In [10]:
dfMetaDadosDecisao.head(3)

,SEQDOCUMENTO,DATAPUBLICACAO,TIPODOCUMENTO,NUMEROREGISTRO,PROCESSO,DATARECEBIMENTO,DATADISTRIBUICAO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,ASSUNTOS
0,309631639,2025-05-05,DECISÃO,202000946481,TP 2695,2020-04-25,2020-04-27,MARIA ISABEL GALLOTTI,NaN,Negando,Prejudicada a ação de #{nome_da_parte} #{campo...,4993
1,310378377,2025-05-05,DECISÃO,202400362531,AREsp 2562174,2024-02-09,2024-02-28,JOÃO OTÁVIO DE NORONHA,NaN,Negando,Conhecido o recurso de #{nome_da_parte} e não-...,"11806, 7752, 7780, 7779"
2,309409286,2025-05-05,DECISÃO,202401080738,REsp 2132967,2024-04-02,2024-04-04,MESSOD AZULAY NETO,NaN,Concedendo,Conhecido o recurso de #{nome_da_parte} e prov...,"3416, 3435"


In [11]:
dfMetaDadosDecisao.drop(columns=['DATAPUBLICACAO','TIPODOCUMENTO','DATARECEBIMENTO','DATADISTRIBUICAO','ASSUNTOS'], inplace=True)
dfMetaDadosDecisao.head(3)

,SEQDOCUMENTO,NUMEROREGISTRO,PROCESSO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA
0,309631639,202000946481,TP 2695,MARIA ISABEL GALLOTTI,NaN,Negando,Prejudicada a ação de #{nome_da_parte} #{campo...
1,310378377,202400362531,AREsp 2562174,JOÃO OTÁVIO DE NORONHA,NaN,Negando,Conhecido o recurso de #{nome_da_parte} e não-...
2,309409286,202401080738,REsp 2132967,MESSOD AZULAY NETO,NaN,Concedendo,Conhecido o recurso de #{nome_da_parte} e prov...


In [12]:
dfDetalheDecisao.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48027 entries, 0 to 48026
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   SEQDOCUMENTO  48027 non-null  int64  
 1   ACORDAO       0 non-null      float64
 2   EMENTA        13 non-null     object 
 3   RELATORIO     0 non-null      float64
 4   VOTO          0 non-null      float64
 5   DECISAO       48011 non-null  object 
dtypes: float64(3), int64(1), object(2)
memory usage: 2.2+ MB


In [13]:
dfDetalheDecisao.head(3)

,SEQDOCUMENTO,ACORDAO,EMENTA,RELATORIO,VOTO,DECISAO
0,309631639,NaN,NaN,NaN,NaN,O presente pedido de contracautela tinha por o...
1,310378377,NaN,NaN,NaN,NaN,Trata-se de agravo em recurso especial interpo...
2,309409286,NaN,NaN,NaN,NaN,Trata-se de recurso especial interposto por MA...


In [14]:
dfDetalheDecisao.drop(columns=['ACORDAO', 'EMENTA', 'RELATORIO', 'VOTO'], inplace=True)
dfDetalheDecisao.head(3)

,SEQDOCUMENTO,DECISAO
0,309631639,O presente pedido de contracautela tinha por o...
1,310378377,Trata-se de agravo em recurso especial interpo...
2,309409286,Trata-se de recurso especial interposto por MA...


##Redução dos registros com o intuito de reduzir o tempo no treinamento e verificar a curva de convergência

###Separar o número de processo pelo tipo (Aresp, HC).

In [15]:
#field = "AREsp 2562174".split(" ")
#print(f"Tipo: '{field[0]}' Processo: '{field[1]}'  ")

dfMetaDadosDecisao['TIPOPROCESSO'] = dfMetaDadosDecisao['PROCESSO'].apply(lambda tipoprocesso: tipoprocesso.split(" ")[0])
dfMetaDadosDecisao['NUMEROPROCESSO'] = dfMetaDadosDecisao['PROCESSO'].apply(lambda numeroprocesso: numeroprocesso.split(" ")[1])
dfMetaDadosDecisao.drop(columns=['PROCESSO'], inplace=True, axis=1)

dfMetaDadosDecisao.head(3)

,SEQDOCUMENTO,NUMEROREGISTRO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,TIPOPROCESSO,NUMEROPROCESSO
0,309631639,202000946481,MARIA ISABEL GALLOTTI,NaN,Negando,Prejudicada a ação de #{nome_da_parte} #{campo...,TP,2695
1,310378377,202400362531,JOÃO OTÁVIO DE NORONHA,NaN,Negando,Conhecido o recurso de #{nome_da_parte} e não-...,AREsp,2562174
2,309409286,202401080738,MESSOD AZULAY NETO,NaN,Concedendo,Conhecido o recurso de #{nome_da_parte} e prov...,REsp,2132967


In [16]:
dfMetaDadosDecisao['TIPOPROCESSO'].value_counts()

,count
TIPOPROCESSO,
AREsp,27718
HC,9123
REsp,6650
CC,1956
RHC,1534
RMS,349
EAREsp,309
Rcl,235
EREsp,122


In [17]:
if ENVIRONMENT == 'COLAB':
  dfMetaDadosDecisao = dfMetaDadosDecisao[dfMetaDadosDecisao['TIPOPROCESSO'] == 'REsp']
  dfMetaDadosDecisao.head(3)


,SEQDOCUMENTO,NUMEROREGISTRO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,TIPOPROCESSO,NUMEROPROCESSO
2,309409286,202401080738,MESSOD AZULAY NETO,NaN,Concedendo,Conhecido o recurso de #{nome_da_parte} e prov...,REsp,2132967
9,309858585,202500230214,TEODORO SILVA SANTOS,NaN,Outros,Prejudicado o recurso de #{nome_da_parte} #{ca...,REsp,2193643
14,308523685,202400909015,BENEDITO GONÇALVES,EDcl,Concedendo,Embargos de Declaração de #{nome_da_parte} Aco...,REsp,2130521


##Sanitização do campo DECISAO

###Remoção de TAG's HTML do campo DECISAO e substituição por espaço em branco

In [18]:
import re

def remove_html_tags(text):
    if isinstance(text, str):
        clean = re.compile('<.*?>')
        return re.sub(clean, ' ', text)
    return text

In [19]:
dfDetalheDecisao['DECISAO'] = dfDetalheDecisao['DECISAO'].apply(remove_html_tags)
display(dfDetalheDecisao.head())

,SEQDOCUMENTO,DECISAO
0,309631639,O presente pedido de contracautela tinha por o...
1,310378377,Trata-se de agravo em recurso especial interpo...
2,309409286,Trata-se de recurso especial interposto por MA...
3,310313783,Em agravo interposto contra decisão do Tribuna...
4,308791735,JOSIMAR TEOTONIO PONTES alega ser vítima de co...


###Remover STOP WORDS


In [20]:
import re

def remove_extra_spaces(text):
    if isinstance(text, str):
        # Replace multiple whitespace characters (spaces, tabs, newlines) with a single space
        text = re.sub(r'\s+', ' ', text).strip()
        # Remove space before commas
        text = re.sub(r'\s+,', ',', text)
        return text
    return text

In [21]:
#Exemplo de uso da função
texto = """   Trata-se , de   habeas   corpus substitutivo    de
recurso          próprio , com pedido de liminar."""
texto = remove_extra_spaces(texto)

print(texto)

Trata-se, de habeas corpus substitutivo de recurso próprio, com pedido de liminar.


In [22]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

stopwords = nltk.corpus.stopwords.words('portuguese')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [23]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def pre_processamento(texto):
    # Ensure texto is a string before processing
    if not isinstance(texto, str):
        return ""

    # Tokenize the text, keeping punctuation as separate tokens
    tokens = word_tokenize(texto, language='portuguese')

    # Get Portuguese stopwords
    stop_words = set(stopwords.words('portuguese'))

    # Filter out stopwords, keeping punctuation tokens and non-alphabetic words
    # A token is kept if it's not a stopword (when lowercased) OR if it's not purely alphabetic (e.g., punctuation, numbers)
    filtered_tokens = [w for w in tokens if w.lower() not in stop_words or not w.isalpha()]

    # Join the tokens back into a string
    texto_limpo = " ".join(filtered_tokens)

    texto_limpo = remove_extra_spaces(texto_limpo)

    return texto_limpo

In [24]:
#Exemplo de uso da função
texto = """Trata-se de habeas corpus substitutivo de recurso próprio, com pedido de liminar, impetrado em favor de LUIS FELIPE SILVA DE ASSIS, em que se aponta com autoridade coatora o Tribunal de Justiça do Estado do Rio de Janeiro.Consta dos autos que o paciente foi condenado à pena de 5 anos e 10 meses de reclusão, em regime semiaberto, mais pagamento de 583 dias-multa, como incurso no art. 33,caput, c.c o 40, VI, da Lei n. 11.343/2006.Em sede recursal, o Tribunal de origem negou provimento ao apelo defensivo"""
texto = pre_processamento(texto)

print(texto)

Trata-se habeas corpus substitutivo recurso próprio, pedido liminar, impetrado favor LUIS FELIPE SILVA ASSIS, aponta autoridade coatora Tribunal Justiça Estado Rio Janeiro.Consta autos paciente condenado pena 5 anos 10 meses reclusão, regime semiaberto, pagamento 583 dias-multa, incurso art . 33, caput, c.c 40, VI, Lei n . 11.343/2006.Em sede recursal, Tribunal origem negou provimento apelo defensivo


In [25]:
dfDetalheDecisao['DECISAO'] = dfDetalheDecisao['DECISAO'].apply(pre_processamento)

###Junção dos Datasets em um único dataset

In [26]:
def get_decisao(seq_documento):
    # Filter dfDetalheAcordaos for the given SEQDOCUMENTO
    decisao_row = dfDetalheDecisao[dfDetalheDecisao['SEQDOCUMENTO'] == seq_documento]

    # Check if the filtered DataFrame is empty
    if not decisao_row.empty:
        # If not empty, return the 'ACORDAO' value
        return decisao_row.head(1)['DECISAO'].iloc[0]
    else:
        # If empty, return None
        return None

def get_tipo_decisao(teor_decisao):
    # Filter dfDetalheAcordaos for the given SEQDOCUMENTO
    tipo_decisao_row = dfTipoDecisao[dfTipoDecisao['DESC_TEOR_DECISAO'] == teor_decisao].head(1)

    # Check if the filtered DataFrame is empty
    if not tipo_decisao_row.empty:
        # If not empty, return the 'ACORDAO' value
        return tipo_decisao_row.head(1)['DECISAO_FAVORAVEL'].iloc[0]
    else:
        # If empty, return None
        return None

In [27]:
dfMetaDadosDecisao['DECISAO']  = dfMetaDadosDecisao['SEQDOCUMENTO'].apply(get_decisao)
dfMetaDadosDecisao['DECISAO_FAVORAVEL']  = dfMetaDadosDecisao['TEOR'].apply(get_tipo_decisao)

In [28]:
dfMetaDadosDecisao.head(3)

,SEQDOCUMENTO,NUMEROREGISTRO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,TIPOPROCESSO,NUMEROPROCESSO,DECISAO,DECISAO_FAVORAVEL
2,309409286,202401080738,MESSOD AZULAY NETO,NaN,Concedendo,Conhecido o recurso de #{nome_da_parte} e prov...,REsp,2132967,Trata-se recurso especial interposto MAYCKON S...,S
9,309858585,202500230214,TEODORO SILVA SANTOS,NaN,Outros,Prejudicado o recurso de #{nome_da_parte} #{ca...,REsp,2193643,"Sobre pretensão recursal, Primeira Seção deste...",S
14,308523685,202400909015,BENEDITO GONÇALVES,EDcl,Concedendo,Embargos de Declaração de #{nome_da_parte} Aco...,REsp,2130521,Trata-se embargos declaração opostos face deci...,S


In [29]:
# Filter rows where 'TEOR' is not empty (NaN) and assign to a new dataframe
dfMetaDadosDecisao_filtered = dfMetaDadosDecisao.dropna(subset=['TEOR']).copy()

# Filter rows where 'DECISAO' is not empty (NaN) and assign to a new dataframe
dfMetaDadosDecisao_filtered = dfMetaDadosDecisao_filtered.dropna(subset=['DECISAO']).copy()

# Filter rows where 'DECISAO_FAVORAVEL is not empty (NaN) and assign to a new dataframe -- decisões neutras não possuem efeito jurídico considerável
dfMetaDadosDecisao_filtered = dfMetaDadosDecisao_filtered.dropna(subset=['DECISAO_FAVORAVEL']).copy()

dfMetaDadosDecisao = dfMetaDadosDecisao_filtered
dfMetaDadosDecisao.head(5)

,SEQDOCUMENTO,NUMEROREGISTRO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,TIPOPROCESSO,NUMEROPROCESSO,DECISAO,DECISAO_FAVORAVEL
2,309409286,202401080738,MESSOD AZULAY NETO,NaN,Concedendo,Conhecido o recurso de #{nome_da_parte} e prov...,REsp,2132967,Trata-se recurso especial interposto MAYCKON S...,S
9,309858585,202500230214,TEODORO SILVA SANTOS,NaN,Outros,Prejudicado o recurso de #{nome_da_parte} #{ca...,REsp,2193643,"Sobre pretensão recursal, Primeira Seção deste...",S
14,308523685,202400909015,BENEDITO GONÇALVES,EDcl,Concedendo,Embargos de Declaração de #{nome_da_parte} Aco...,REsp,2130521,Trata-se embargos declaração opostos face deci...,S
19,309693182,202201927262,OTÁVIO DE ALMEIDA TOLEDO (DESEMBARGADOR CONVOC...,NaN,Concedendo,Conhecido em parte o recurso de #{nome_da_part...,REsp,2009829,Trata-se recurso especial interposto LEONARDO ...,S
41,310045350,202500842532,BENEDITO GONÇALVES,NaN,Não Conhecendo,Não conhecido o recurso de #{nome_da_parte} #{...,REsp,2202255,Trata-se recurso especial interposto AUTOPOSTO...,N


In [30]:
dfMetaDadosDecisao.head()

,SEQDOCUMENTO,NUMEROREGISTRO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,TIPOPROCESSO,NUMEROPROCESSO,DECISAO,DECISAO_FAVORAVEL
2,309409286,202401080738,MESSOD AZULAY NETO,NaN,Concedendo,Conhecido o recurso de #{nome_da_parte} e prov...,REsp,2132967,Trata-se recurso especial interposto MAYCKON S...,S
9,309858585,202500230214,TEODORO SILVA SANTOS,NaN,Outros,Prejudicado o recurso de #{nome_da_parte} #{ca...,REsp,2193643,"Sobre pretensão recursal, Primeira Seção deste...",S
14,308523685,202400909015,BENEDITO GONÇALVES,EDcl,Concedendo,Embargos de Declaração de #{nome_da_parte} Aco...,REsp,2130521,Trata-se embargos declaração opostos face deci...,S
19,309693182,202201927262,OTÁVIO DE ALMEIDA TOLEDO (DESEMBARGADOR CONVOC...,NaN,Concedendo,Conhecido em parte o recurso de #{nome_da_part...,REsp,2009829,Trata-se recurso especial interposto LEONARDO ...,S
41,310045350,202500842532,BENEDITO GONÇALVES,NaN,Não Conhecendo,Não conhecido o recurso de #{nome_da_parte} #{...,REsp,2202255,Trata-se recurso especial interposto AUTOPOSTO...,N


### Junção dos campos DESCRICAOMONOCRATICA com DECISAO
Estudar a possibilidade de dar maior atenção à decisão monocrática.

In [31]:
dfMetaDadosDecisao.head(3)

,SEQDOCUMENTO,NUMEROREGISTRO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,TIPOPROCESSO,NUMEROPROCESSO,DECISAO,DECISAO_FAVORAVEL
2,309409286,202401080738,MESSOD AZULAY NETO,NaN,Concedendo,Conhecido o recurso de #{nome_da_parte} e prov...,REsp,2132967,Trata-se recurso especial interposto MAYCKON S...,S
9,309858585,202500230214,TEODORO SILVA SANTOS,NaN,Outros,Prejudicado o recurso de #{nome_da_parte} #{ca...,REsp,2193643,"Sobre pretensão recursal, Primeira Seção deste...",S
14,308523685,202400909015,BENEDITO GONÇALVES,EDcl,Concedendo,Embargos de Declaração de #{nome_da_parte} Aco...,REsp,2130521,Trata-se embargos declaração opostos face deci...,S


##Análise de dados após tratamento

In [32]:
dfMetaDadosDecisao['DECISAOCOMPLETA'] = dfMetaDadosDecisao['DESCRICAOMONOCRATICA'] + ' -- ' + dfMetaDadosDecisao['DECISAO']

###Verificando distribuição de dados entre as decisões favoráveis ou não ao réu

####Consulta direta no tipo de decisão (Favorável ou não)

In [33]:
#dfMetaDadosDecisao['DECISAOCOMPLETA'][283]

In [34]:
dfMetaDadosDecisao_filtered[dfMetaDadosDecisao_filtered['DECISAO_FAVORAVEL']  == 'N'].head(5)

,SEQDOCUMENTO,NUMEROREGISTRO,NM_MINISTRO,RECURSO,TEOR,DESCRICAOMONOCRATICA,TIPOPROCESSO,NUMEROPROCESSO,DECISAO,DECISAO_FAVORAVEL,DECISAOCOMPLETA
41,310045350,202500842532,BENEDITO GONÇALVES,NaN,Não Conhecendo,Não conhecido o recurso de #{nome_da_parte} #{...,REsp,2202255,Trata-se recurso especial interposto AUTOPOSTO...,N,Não conhecido o recurso de #{nome_da_parte} #{...
51,309749316,202500351780,OTÁVIO DE ALMEIDA TOLEDO (DESEMBARGADOR CONVOC...,NaN,Negando,Conhecido o recurso de #{nome_da_parte} e não-...,REsp,2196863,Trata-se recurso especial interposto MINISTÉRI...,N,Conhecido o recurso de #{nome_da_parte} e não-...
66,310055128,202500308857,TEODORO SILVA SANTOS,NaN,Não Conhecendo,Não conhecido o recurso de #{nome_da_parte} #{...,REsp,2195009,Trata-se recurso especial interposto POLIMETAL...,N,Não conhecido o recurso de #{nome_da_parte} #{...
82,309989594,202500016479,TEODORO SILVA SANTOS,NaN,Não Conhecendo,Não conhecido o recurso de #{nome_da_parte} #{...,REsp,2190839,Trata-se recurso especial interposto ESTADO PA...,N,Não conhecido o recurso de #{nome_da_parte} #{...
86,308489875,202400980401,BENEDITO GONÇALVES,NaN,Não Conhecendo,Não conhecido o recurso de #{nome_da_parte} #{...,REsp,2131684,Trata-se recurso especial interposto PROMON EN...,N,Não conhecido o recurso de #{nome_da_parte} #{...


In [35]:
dfMetaDadosDecisao_filtered.shape

(6563, 11)

####Verificando distribuição dos dados propriamente dita

In [36]:
dataset = dfMetaDadosDecisao

total   = dataset.shape[0]
label_s = dfMetaDadosDecisao_filtered[dataset['DECISAO_FAVORAVEL']  == 'S'].shape[0]
label_n = dfMetaDadosDecisao_filtered[dataset['DECISAO_FAVORAVEL']  == 'N'].shape[0]

print(f'TOTAL  : {total}')
print(f'LABEL S: {label_s} - % {round(label_s/total*100, 2)}')
print(f'LABEL N: {label_n} - % {round(label_n/total*100, 2)}')

TOTAL  : 6563
LABEL S: 2767 - % 42.16
LABEL N: 3796 - % 57.84


##Preparação dos dados para Treinamento

###Separando entre dados de treinamento e teste

In [37]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = dataset.drop('DECISAO_FAVORAVEL', axis=1)
y = dataset['DECISAO_FAVORAVEL']

# Split the data into training and testing sets, stratifying by 'DECISAO_FAVORAVEL'
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.35, random_state=42, stratify=y)

print("Training set shape (X_train, y_train):", X_train.shape, y_train.shape)
print("Testing set shape (X_test, y_test):", X_test.shape, y_test.shape)

# Display the distribution of 'DECISAO_FAVORAVEL' in the original dataset, training set, and testing set
print("\nOriginal dataset 'DECISAO_FAVORAVEL' distribution:")
display(y.value_counts(normalize=True))

print("\nTraining set 'DECISAO_FAVORAVEL' distribution:")
display(y_train.value_counts(normalize=True))

print("\nTesting set 'DECISAO_FAVORAVEL' distribution:")
display(y_test.value_counts(normalize=True))

Training set shape (X_train, y_train): (4265, 10) (4265,)
Testing set shape (X_test, y_test): (2298, 10) (2298,)

Original dataset 'DECISAO_FAVORAVEL' distribution:


,proportion
DECISAO_FAVORAVEL,
N,0.578394
S,0.421606



Training set 'DECISAO_FAVORAVEL' distribution:


,proportion
DECISAO_FAVORAVEL,
N,0.578429
S,0.421571



Testing set 'DECISAO_FAVORAVEL' distribution:


,proportion
DECISAO_FAVORAVEL,
N,0.578329
S,0.421671


In [38]:
#dataframeGhost['gasper'] = 0

###Selecionando modelo pre-treinado e realizando tokenização

In [39]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification # Or BertTokenizer, BertForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2) # Use AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, do_lower_case=False)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-large-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
print(f"Model's hidden size (embedding dimensionality): {model.config.hidden_size}")
print(f"Model's maximum position embeddings (max sequence length): {model.config.max_position_embeddings}")

Model's hidden size (embedding dimensionality): 1024
Model's maximum position embeddings (max sequence length): 512


In [41]:
from transformers import DataCollatorWithPadding
from datasets import Dataset # Import Dataset

# Pad to the longest sequence in the batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
max_length    = model.config.max_position_embeddings

def preprocess_function(batches):
   """Tokenize input data"""
   return tokenizer(batches["DECISAO"], max_length=max_length, truncation=True) # Use "DECISAO" as the column name

# Convert pandas Series to Hugging Face Dataset
train_dataset = Dataset.from_pandas(X_train)
test_dataset = Dataset.from_pandas(X_test)

# Convert labels to integers
'''
The Hugging Face Trainer usually requires integer labels for classification.
I will convert the string labels ('S' and 'N') to integers (e.g., 0 and 1) before
adding them to the dataset.
'''
label_map = {'N': 0, 'S': 1}
y_train_encoded = y_train.map(label_map)
y_test_encoded = y_test.map(label_map)

# Add labels to the datasets
train_dataset = train_dataset.add_column("labels", y_train_encoded.tolist())
test_dataset = test_dataset.add_column("labels", y_test_encoded.tolist())

# Tokenize train/test data using the Dataset map method
tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/4265 [00:00<?, ? examples/s]

Map:   0%|          | 0/2298 [00:00<?, ? examples/s]

First, let's inspect the names of the layers in your model. This will help you decide which layers you want to freeze. Running the following cell will print each layer's name and its index.

In [42]:
# Print layer names
# The index can help you identify which layers correspond to earlier blocks (e.g., embedding, early encoder layers)
i = 0
for name, param in model.named_parameters():
  i = i + 1
  print(f"index: {i} {name}")

index: 1 bert.embeddings.word_embeddings.weight
index: 2 bert.embeddings.position_embeddings.weight
index: 3 bert.embeddings.token_type_embeddings.weight
index: 4 bert.embeddings.LayerNorm.weight
index: 5 bert.embeddings.LayerNorm.bias
index: 6 bert.encoder.layer.0.attention.self.query.weight
index: 7 bert.encoder.layer.0.attention.self.query.bias
index: 8 bert.encoder.layer.0.attention.self.key.weight
index: 9 bert.encoder.layer.0.attention.self.key.bias
index: 10 bert.encoder.layer.0.attention.self.value.weight
index: 11 bert.encoder.layer.0.attention.self.value.bias
index: 12 bert.encoder.layer.0.attention.output.dense.weight
index: 13 bert.encoder.layer.0.attention.output.dense.bias
index: 14 bert.encoder.layer.0.attention.output.LayerNorm.weight
index: 15 bert.encoder.layer.0.attention.output.LayerNorm.bias
index: 16 bert.encoder.layer.0.intermediate.dense.weight
index: 17 bert.encoder.layer.0.intermediate.dense.bias
index: 18 bert.encoder.layer.0.output.dense.weight
index: 19 ber

After inspecting the layer names and their indices, you can decide how many initial layers (or specific layers) you want to freeze. Freezing a layer means setting its `requires_grad` attribute to `False`, so its parameters won't be updated during backpropagation.

In [43]:
print(f"Freezing the first {num_layers_to_freeze} layers of the model...")
for index, (name, param) in enumerate(model.named_parameters()):
    if index < num_layers_to_freeze:
        param.requires_grad = False
    else:
        # Ensure subsequent layers are trainable if you only want to freeze initial ones
        param.requires_grad = True
    # print(f"Layer: {name}, Requires grad: {param.requires_grad}") # Uncomment to verify

print("Layers freezing complete. You can now proceed with training.")

Freezing the first 165 layers of the model...
Layers freezing complete. You can now proceed with training.


In [44]:
#Show diferences between standard dataset and pandas dataset.
train_dataset['labels']

Column([1, 0, 0, 1, 1])

In [45]:
%%capture
if not install_evaluate:
  install_evaluate = True
  %pip install evaluate

###Definindo método para averiguar a qualidade do modelo (métrica de avaliação)

In [46]:
import numpy as np
import evaluate

def compute_metrics(eval_pred):
    """Calculate F1 score"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    load_f1 = evaluate.load("f1")
    f1 = load_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"f1": f1}

In [47]:
import sys

def get_avg_epoch_time(log_history: list) -> float | None:
    """
    Calcula o tempo médio por época (em minutos) a partir do
    log_history do Hugging Face Trainer.

    Args:
        log_history (list): Uma lista de dicionários, que é o
                            trainer.state.log_history.

    Returns:
        float: O tempo médio por época em minutos.
        None: Se os dados necessários não forem encontrados no histórico.
    """

    # 1. Encontrar o log de resumo do treino.
    #    Iteramos de trás para frente, pois o resumo é sempre um dos
    #    últimos itens logados.
    train_summary = None
    for log in reversed(log_history):
        if 'train_runtime' in log:
            train_summary = log
            break  # Encontramos o log que queremos

    # 2. Verificar se encontramos o log
    if train_summary is None:
        print("Erro: 'train_runtime' não foi encontrado no histórico de logs.", file=sys.stderr)
        return None

    # 3. Extrair os dados necessários
    # Usamos .get() para evitar erros caso a chave não exista
    total_seconds = train_summary.get('train_runtime')
    total_epochs = train_summary.get('epoch')

    # 4. Validar os dados
    if total_seconds is None or total_epochs is None:
        print("Erro: 'train_runtime' ou 'epoch' estão faltando no log de resumo.", file=sys.stderr)
        return None

    if total_epochs == 0:
        print("Erro: O número total de épocas é zero. Divisão por zero evitada.", file=sys.stderr)
        return None

    # 5. Calcular e retornar o tempo em minutos
    seconds_per_epoch = total_seconds / total_epochs
    minutes_per_epoch = seconds_per_epoch / 60

    return minutes_per_epoch

In [48]:
# X_train['Exception Please']

In [49]:
# # Print layer names
# i = 0
# for name, param in model.named_parameters():
#   i = i +1
#   print(f"index: {i} {name}")


In [50]:
# Encoder block 10 starts at index 165 and
# we freeze everything before that block
# for index, (name, param) in enumerate(model.named_parameters()):
#     if index < 165:
#         param.requires_grad = False

###Preparação do modelo

In [51]:
%%capture
!pip install tf_keras

In [52]:
# %%capture
# !pip install 'accelerate>=0.26.0'

In [53]:
# %%capture
# !pip install transformers[torch]

In [54]:
# This is the line that gets the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cuda


In [55]:
from transformers import TrainingArguments, Trainer
import torch # Import torch

EPOCHS = 3
APPROACH_EXECUTION = 'step' #'epoch'

# Training arguments for parameter tuning
if APPROACH_EXECUTION == 'epoch':
    training_args = TrainingArguments(
       "model",
       learning_rate= 2e-5,
       per_device_train_batch_size=16,
       per_device_eval_batch_size=16,
       num_train_epochs=EPOCHS,
       weight_decay=0.01,
       save_strategy="epoch",
       logging_strategy="epoch",
       report_to="none",
    )
else:#step
    training_args = TrainingArguments(
       "model",
       learning_rate= 2e-5,
       per_device_train_batch_size=32,
       per_device_eval_batch_size=32,
       num_train_epochs=EPOCHS,
       weight_decay=0.01,
       save_strategy="steps",
       eval_steps=100, # Evaluate every 100 steps
       logging_steps=100, # Log every 100 steps
       report_to="none",
       metric_for_best_model="f1", # Metric to monitor for best model selection
       greater_is_better=True, # A higher F1 score is better
    )

In [56]:
# Trainer which executes the training process
trainer = Trainer(
   model=model,
   args=training_args,
   train_dataset=tokenized_train,
   eval_dataset=tokenized_test,
   tokenizer=tokenizer,
   data_collator=data_collator,
   compute_metrics=compute_metrics,
)

/tmp/ipython-input-2254602729.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


###Treinamento

In [57]:
import datetime
import time

def format_duration(td: datetime.timedelta) -> str:
    """Formata um objeto timedelta em HH:MM:SS."""
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    # Retorna no formato 00:00:00
    return f"{hours:02}:{minutes:02}:{seconds:02}"

In [ ]:
start_time = datetime.datetime.now()

start_time_str = start_time.strftime("%H:%M:%S")
print(f"✅ Processo iniciado em: {start_time_str}")

trainer.train()

end_time = datetime.datetime.now()
end_time_str = end_time.strftime("%H:%M:%S")
print(f"Processo finalizado às: {end_time_str}")

# --- 4. Calcular e imprimir a duração ---
duration = end_time - start_time

print("\n--- Resumo da Duração ---")
# Imprime o objeto timedelta (pode incluir microssegundos)
print(f"Duração (Objeto): {duration}")

# Imprime o total de segundos
print(f"Duração (Segundos): {duration.total_seconds():.2f} s")

# Imprime no formato HH:MM:SS
print(f"Duração (Formatada): {format_duration(duration)}")
print(f'Media de execucao por epoca: {round(get_avg_epoch_time(trainer.state.log_history),2) } min.')

✅ Processo iniciado em: 22:35:54


Step,Training Loss


##Avaliação do modelo

In [ ]:
trainer.state.log_history

In [ ]:
###Salvar modelo para execucao

In [ ]:
import os

# Define the directory to save the model and tokenizer
output_dir = '/dados_abertos_stj/saved_models/model_with_stop_words'

# Create the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Save the trained model
model.save_pretrained(output_dir)

# Save the tokenizer
tokenizer.save_pretrained(output_dir)

print(f"Model and tokenizer saved to {output_dir}")

###Curva de convergência

In [ ]:
import matplotlib.pyplot as plt

# Get the training history
history = trainer.state.log_history

# Extract loss and learning rate values and steps
steps = []
loss_values = []
learning_rates = []

for entry in history:
    if 'loss' in entry:
        steps.append(entry['step'])
        loss_values.append(entry['loss'])
    if 'learning_rate' in entry:
        learning_rates.append(entry['learning_rate'])

# Ensure learning_rates list has the same length as steps/loss_values
# This might be necessary if learning_rate is not logged at every step
# A simple forward fill or interpolation could be used if needed,
# but for now, assume they are logged at similar intervals.
# If the lengths differ significantly, further investigation of the log history structure is needed.
# For this example, we'll assume they align closely or learning_rate entries are sufficient.
# A more robust approach would match steps between loss and learning_rate entries.

# Simple check: if lengths differ, we might need a more sophisticated approach
if len(steps) != len(learning_rates):
    print("Warning: Length of loss and learning rate entries in history differ.")
    # Depending on logging frequency, you might need to align these based on 'step'

# Plot the convergence curve and learning rate
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('Steps')
ax1.set_ylabel('Training Loss', color=color)
ax1.plot(steps, loss_values, color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'tab:blue'
ax2.set_ylabel('Learning Rate', color=color)  # we already handled the x-label with ax1
ax2.plot(steps, learning_rates, color=color)
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.title('Training Loss and Learning Rate over Steps')
plt.show()

###Avaliação propriamenta dita

In [ ]:
trainer.evaluate()

###Matriz de Confusão

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

# Get predictions from the trainer
predictions = trainer.predict(tokenized_test)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# --- Seus dados (simulados para o exemplo funcionar) ---
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = tokenized_test["labels"]

# --- Novas Métricas ---
print("=" * 30)
print("MÉTRICAS DE AVALIAÇÃO")
print("=" * 30)

# Accuracy (Acurácia)
accuracy = accuracy_score(true_labels, predicted_labels)
print(f"Accuracy (Acurácia): {accuracy:.4f}")

# Precision (Precisão)
precision = precision_score(true_labels, predicted_labels)
print(f"Precision (Precisão): {precision:.4f}")

# Recall (Revocação / Sensibilidade / True Positive Rate - TPR)
recall = recall_score(true_labels, predicted_labels)
print(f"Recall (Sensibilidade/TPR): {recall:.4f}")

# True Negative Rate (TNR) / Specificity (Especificidade)
# Esta é a "revocação" da classe negativa (label = 0)
tnr = recall_score(true_labels, predicted_labels, pos_label=0)
print(f"True Negative Rate (Especificidade/TNR): {tnr:.4f}")

# --- ADICIONADO AQUI ---
# False Positive Rate (FPR)
# É o complemento da Especificidade (FPR = 1 - TNR)
fpr = 1 - tnr
print(f"False Positive Rate (FPR): {fpr:.4f}")
# --- FIM DA ADIÇÃO ---

# F1-Score
f1 = f1_score(true_labels, predicted_labels)
print(f"F1-Score: {f1:.4f}")

print("-" * 30)

# --- Plot da Matriz de Confusão ---

# Generate the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Display the confusion matrix with desired label order
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.show()

###Curva ROC

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import auc
from sklearn.metrics import roc_curve

# Get the prediction scores (probabilities) for the positive class
# For binary classification with logits, you typically apply a sigmoid or softmax
# Here, assuming logits, we'll take the exponent of the positive class logit
# If your model outputs probabilities directly, adjust this line
prediction_scores = np.exp(predictions.predictions[:, 1]) / np.sum(np.exp(predictions.predictions), axis=1)

# Get the true labels
true_labels = tokenized_test["labels"]

# Calculate the ROC curve points
fpr, tpr, thresholds = roc_curve(true_labels, prediction_scores)

# print("False Positive Rate (FPR):", fpr)
# print("True Positive Rate (TPR):", tpr)
# print("Thresholds:", thresholds)

# Calculate the Area Under the Curve (AUC)
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys

def plot_training_history(log_history: list):
    """
    Processa o log_history do Hugging Face Trainer e plota os gráficos
    de perda (loss) e métricas de avaliação.

    Args:
        log_history (list): A lista de dicionários de trainer.state.log_history.
    """

    # 1. Separar logs de treino e avaliação
    train_logs = []
    eval_logs = []

    for log in log_history:
        if 'eval_loss' in log:
            # Este é um log de avaliação
            eval_logs.append(log)
        elif 'loss' in log:
            # Este é um log de treino (passo a passo)
            train_logs.append(log)
        # Ignoramos os logs de resumo final (ex: 'train_runtime')

    if not train_logs and not eval_logs:
        print("Histórico de log está vazio ou não contém logs de treino/avaliação.", file=sys.stderr)
        return

    # 2. Criar DataFrames
    train_df = pd.DataFrame(train_logs)
    eval_df = pd.DataFrame(eval_logs)

    # 3. Determinar o eixo X (preferir 'epoch', mas usar 'step' se 'epoch' não estiver)
    x_axis_train = 'epoch' if 'epoch' in train_df.columns else 'step'
    x_axis_eval = 'epoch' if 'epoch' in eval_df.columns else 'step'

    # 4. Encontrar a primeira métrica de avaliação (ex: 'eval_f1', 'eval_accuracy')
    metric_name = None
    if not eval_df.empty:
        for col in eval_df.columns:
            if col.startswith('eval_') and col not in ['eval_loss', 'eval_runtime',
                                                       'eval_samples_per_second',
                                                       'eval_steps_per_second']:
                metric_name = col
                break  # Encontramos a primeira métrica

    # 5. Criar os plots

    # Determinar quantos subplots precisamos
    num_plots = 1 + (1 if metric_name else 0)

    if num_plots == 2:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    else:
        fig, ax1 = plt.subplots(1, 1, figsize=(9, 6))
        ax2 = None # Sem segundo eixo

    # --- Gráfico 1: Perda (Loss) ---
    if not train_df.empty:
        ax1.plot(train_df[x_axis_train], train_df['loss'], label='Training Loss', alpha=0.7)
    if not eval_df.empty:
        ax1.plot(eval_df[x_axis_eval], eval_df['eval_loss'], label='Validation Loss',
                 marker='o', markersize=5, linestyle='--')

    ax1.set_title('Training vs. Validation Loss')
    ax1.set_xlabel('Epoch' if x_axis_train == 'epoch' else 'Step')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, linestyle=':', alpha=0.6)

    # --- Gráfico 2: Métrica (se existir) ---
    if ax2 and metric_name:
        if not eval_df.empty:
            ax2.plot(eval_df[x_axis_eval], eval_df[metric_name],
                     label=f'Validation {metric_name}',
                     marker='o', markersize=5, linestyle='--', color='green')

        ax2.set_title(f'Validation Metric ({metric_name})')
        ax2.set_xlabel('Epoch' if x_axis_eval == 'epoch' else 'Step')
        ax2.set_ylabel(metric_name)
        ax2.legend()
        ax2.grid(True, linestyle=':', alpha=0.6)

    # Salvar e mostrar o gráfico
    plt.tight_layout()
    plot_filename = 'training_history.png'
    plt.savefig(plot_filename)

    print(f"✅ Gráficos de histórico de treino salvos em '{plot_filename}'")

In [ ]:
plot_training_history(trainer.state.log_history)